# EDA — LLD-MMRI (W2 ngày 2)

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Notebook này là **lớp mỏng** gọi vào `src/` (AGENTS.md §4) — không chứa logic.
Mọi tính toán nằm ở `src/data/eda.py` và `src/data/geometry_gate.py` để test được
và tái dùng ở bước tiền xử lý.

**Đầu ra cần có (T2.1 trong `docs/W2_plan.md`):**
1. Phân bố 7 lớp — đối chiếu phân bố official trong PDF challenge
2. Spacing / thickness theo từng pha
3. Tỉ lệ ca thiếu pha
4. Thống kê kích thước bbox lesion
5. Khuyến nghị crop size
6. **GATE GEOMETRY (bắt buộc)** — không đạt thì DỪNG, không crop theo bbox

Cách chạy trên Kaggle: xem [`docs/KAGGLE_WORKFLOW.md`](../docs/KAGGLE_WORKFLOW.md).

## 0. Bootstrap

Chạy được ở cả hai nơi: **Kaggle** (clone repo + mount data) và **local** (nếu đã
đặt `LLDMMRI_DATA_ROOT`). Không hardcode path — lấy từ env/config.

In [ ]:
import os
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        os.system(f"git clone -q https://github.com/hdtruong802/liver-mri-3d-classifier.git {REPO}")
    sys.path.insert(0, str(REPO))
    os.environ.setdefault("LLDMMRI_DATA_ROOT", "/kaggle/input/lldmmridataset")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO))

from src.utils.io import load_yaml, resolve_data_root  # noqa: E402
from src.utils.seed import set_seed  # noqa: E402

CONFIG = load_yaml(REPO / "configs" / "data.yaml")
set_seed(CONFIG["seed"])

DATA_ROOT = resolve_data_root(CONFIG)
ANNOTATION_PATH = DATA_ROOT / CONFIG["annotation_rel"]
IMAGES_DIR = DATA_ROOT / CONFIG["images_rel"]
PHASES = CONFIG["phases"]
PHASE_TOKENS = [p["file"] for p in PHASES]

print("on kaggle :", ON_KAGGLE)
print("repo      :", REPO)
print("data root :", DATA_ROOT)
print("annotation:", ANNOTATION_PATH, "| tồn tại:", ANNOTATION_PATH.exists())
print("images    :", IMAGES_DIR, "| tồn tại:", IMAGES_DIR.exists())

In [ ]:
from src.data.annotation import Annotation

ann = Annotation(ANNOTATION_PATH)
print(f"số bệnh nhân: {len(ann)}")
print(f"số lớp      : {len(ann.class_to_index)}")
print("map lớp     :", ann.class_to_index)

## 1. Phân bố 7 lớp

**Đối chiếu bắt buộc** với phân bố official (PDF challenge p.10):
HCC 157 · u máu 79 · ICC 58 · áp-xe 54 · nang 53 · di căn 51 · FNH 46 = 498.

Lệch ⇒ bản dữ liệu không toàn vẹn ⇒ dừng, kiểm tra lại nguồn.

In [ ]:
from src.data.eda import class_distribution, format_class_distribution

dist = class_distribution(ann)
print(format_class_distribution(dist))

OFFICIAL = {6: 157, 0: 79, 1: 58, 2: 54, 4: 53, 3: 51, 5: 46}
mismatch = {k: (dist[k], OFFICIAL[k]) for k in OFFICIAL if dist.get(k) != OFFICIAL[k]}
print()
print("KHỚP official" if not mismatch else f"LỆCH official: {mismatch}")

In [ ]:
import matplotlib.pyplot as plt

from src.data.taxonomy import SHORT_NAMES

items = sorted(dist.items(), key=lambda kv: -kv[1])
labels = [SHORT_NAMES[k] for k, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(labels, values, color="#4a6fa5")
ax.set_ylabel("số bệnh nhân")
ax.set_title("Phân bố 7 lớp tổn thương (n=498)")
for i, v in enumerate(values):
    ax.text(i, v + 2, str(v), ha="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Spacing / thickness theo từng pha

PDF challenge (p.9) nói các thì chụp **khác geometry**: non-contrast coronal,
DWI matrix 132×116 (thô), T1 spacing 2mm vs T2 1mm, đa máy 1.5T/3T.
Số liệu dưới đây xác nhận điều đó trên dữ liệu thật ⇒ **registration là bắt buộc**,
không phải tuỳ chọn.

In [ ]:
from src.data.eda import geometry_summary_by_phase, phase_geometry

geoms = phase_geometry(ann)
summary = geometry_summary_by_phase(geoms)

print(f"{'pha':<12} {'px_x p50':>9} {'px_y p50':>9} {'sl_sp p50':>10} {'sl_th p50':>10}")
for phase_cfg in PHASES:
    name = phase_cfg["name"]
    s = summary.get(name)
    if not s:
        continue
    print(
        f"{name:<12} {s['pixel_spacing_x']['p50']:>9.3f} {s['pixel_spacing_y']['p50']:>9.3f} "
        f"{s['slice_spacing']['p50']:>10.2f} {s['slice_thickness']['p50']:>10.2f}"
    )

In [ ]:
# Dải biến thiên trong từng pha — rộng = không đồng nhất giữa các máy/ca.
for phase_cfg in PHASES:
    name = phase_cfg["name"]
    s = summary.get(name)
    if not s:
        continue
    px = s["pixel_spacing_x"]
    sl = s["slice_spacing"]
    print(
        f"{name:<12} pixel_x [{px['min']:.3f} .. {px['max']:.3f}]   "
        f"slice_spacing [{sl['min']:.2f} .. {sl['max']:.2f}]"
    )

## 3. Ca thiếu pha

Cần **file ảnh thật** ⇒ chỉ chạy được trên Kaggle (hoặc local đã tải data).
Kết quả quyết định chiến lược ở T2.2: loại ca thiếu pha, hay zero-fill + mask?

In [ ]:
from src.data.eda import missing_phase_report
from src.data.images import scan_image_index

if IMAGES_DIR.exists():
    image_index = scan_image_index(IMAGES_DIR, CONFIG["image_suffixes"])
    report = missing_phase_report(ann, image_index, PHASE_TOKENS)
    print(f"tổng      : {report.n_patients}")
    print(f"đủ 8 pha  : {report.n_complete}")
    print(f"thiếu pha : {report.n_incomplete}")
    print()
    for token, n in report.missing_by_phase.items():
        if n:
            print(f"  thiếu {token:<10}: {n} ca")
    if report.incomplete_patients:
        print()
        print("10 ca đầu thiếu pha:")
        for pid, missing in report.incomplete_patients[:10]:
            print(f"  {pid}: {missing}")
else:
    image_index = {}
    print("BỎ QUA: chưa có thư mục ảnh (chạy trên Kaggle để có kết quả này)")

## 4. Kích thước bbox lesion → khuyến nghị crop size

Bản dữ liệu **không có patch cắt sẵn** ⇒ phải crop từ full-volume theo bbox.
Cần biết crop 96×96×48 (Spec Sheet) có phủ hết lesion không, hay phải chỉnh.

In [ ]:
from src.data.eda import bbox_stats, recommend_crop_size

REF_PHASE = "C+V"  # pha portal-venous — pha tham chiếu khi registration (Spec Sheet §2)

stats = bbox_stats(ann, REF_PHASE)
print(f"số ca có bbox ở pha {REF_PHASE}: {len(stats)}")

for field_name, values in [
    ("width_mm", [s.width_mm for s in stats]),
    ("height_mm", [s.height_mm for s in stats]),
    ("depth_mm", [s.depth_mm for s in stats]),
    ("depth_slices", [float(s.depth_slices) for s in stats]),
]:
    ordered = sorted(values)
    p50 = ordered[len(ordered) // 2]
    p95 = ordered[int(0.95 * (len(ordered) - 1))]
    print(f"  {field_name:<13} p50={p50:7.1f}  p95={p95:7.1f}  max={ordered[-1]:7.1f}")

In [ ]:
rec = recommend_crop_size(stats, target_spacing=(1.5, 1.5, 3.0), margin=1.3)
print("khuyến nghị crop (target spacing 1.5×1.5×3.0mm, margin 1.3, phủ p95):")
for k, v in rec.items():
    print(f"  {k:<26}: {v}")

print()
print("Spec Sheet chốt 96×96×48 →", end=" ")
if rec:
    ok = rec["voxels_x"] <= 96 and rec["voxels_y"] <= 96 and rec["voxels_z"] <= 48
    print("ĐỦ phủ p95" if ok else "KHÔNG đủ, cân nhắc chỉnh ở T2.2")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (title, values) in zip(
    axes,
    [
        ("bề rộng (mm)", [s.width_mm for s in stats]),
        ("bề cao (mm)", [s.height_mm for s in stats]),
        ("bề sâu (mm)", [s.depth_mm for s in stats]),
    ],
    strict=False,
):
    ax.hist(values, bins=40, color="#4a6fa5")
    ax.set_title(title, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Kích thước lesion, pha {REF_PHASE}", fontsize=11)
plt.tight_layout()
plt.show()

## 5. GATE GEOMETRY — bắt buộc ⚠️

**Vì sao:** bản dữ liệu là `wanglab/LLD-MMRI-MedSAM2` — bản đóng gói lại cho
segmentation. Nó *có thể* đã resample/reorient ảnh trong khi annotation vẫn giữ
toạ độ gốc. Nếu vậy, **crop theo bbox sẽ cắt nhầm chỗ và mọi kết quả sau đều vô nghĩa**.

Gate 3 tầng:
1. spacing header NIfTI vs annotation
2. bbox có nằm trong biên ảnh không (+ xác định axis order)
3. **mắt người**: overlay bbox lên slice → có trúng tổn thương không

**Không đạt ⇒ DỪNG. Không đi tiếp sang tiền xử lý.**

In [ ]:
from src.data.geometry_gate import run_gate

if image_index:
    sample_pids = ann.patient_ids()[:5]
    gate = run_gate(sample_pids, ann, image_index, PHASES)
    print(gate.summary())
else:
    gate = None
    print("BỎ QUA: cần ảnh thật (chạy trên Kaggle)")

In [ ]:
# Tầng 3: overlay bbox lên slice giữa lesion — MẮT NGƯỜI xác nhận trúng u.
from matplotlib import patches

from src.data.geometry_gate import bbox_overlay_slice

if image_index:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for ax, pid in zip(axes, ann.patient_ids()[:3], strict=False):
        sl, box, mid = bbox_overlay_slice(pid, REF_PHASE, "C+V", ann, image_index)
        ax.imshow(sl.T, cmap="gray", origin="lower")
        ax.add_patch(
            patches.Rectangle(
                (box.x_min, box.y_min),
                box.x_max - box.x_min,
                box.y_max - box.y_min,
                linewidth=1.6,
                edgecolor="#e8590c",
                facecolor="none",
            )
        )
        ax.set_title(f"{pid} · slice {mid}", fontsize=9)
        ax.axis("off")
    fig.suptitle("GATE tầng 3 — hộp có trùng tổn thương không?", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("BỎ QUA: cần ảnh thật (chạy trên Kaggle)")

## 6. Kết luận → quyết định cho T2.2

Điền sau khi chạy trên Kaggle, rồi ghi vào `configs/preprocess.yaml` (comment) + WORKLOG:

| Câu hỏi | Kết quả | Quyết định |
|---|---|---|
| Phân bố lớp khớp official? | _(mục 1)_ | |
| Gate geometry đạt? | _(mục 5)_ | **không đạt ⇒ dừng** |
| Axis order của ảnh | _(mục 5)_ | |
| Crop size | _(mục 4)_ | giữ 96×96×48 hay chỉnh? |
| Ca thiếu pha | _(mục 3)_ | loại hay zero-fill + mask? |
| Spacing đích | _(mục 2)_ | giữ 1.5×1.5×3.0mm? |
| N4 ở v0 | — | mặc định **off** (chậm); bật sau nếu cần |